# Phase 2A: Neutral Generation

Run all 9 models on 300 neutral prompts (no trait elicitation).
Responses are saved to `results/phase2/responses/{model_key}_responses.jsonl`.
Scoring (Phase 2B) is done separately, locally, without GPU.

**Workflow:**
1. Section 0 — Config & setup (run once)
2. Section 1 — Utility functions (imported from `scripts/helpers.py`)
3. Section 2 — TEST cell (3 queries, base model) → verify before full run
4. Section 2 — FULL RUN cell (all 9 models × 300 queries, batched, resumable)
5. Section 3 — Response audit (print samples from any completed model)

## Section 0 — Config & Setup

In [1]:
import gc
import json
import logging
import sys
import time
from pathlib import Path

import torch

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

# ── Paths ──────────────────────────────────────────────────────────────────────
# Change to Path(".") for local runs (analysis only — no GPU needed for Section 3)
BASE_DIR = Path("/workspace")

sys.path.insert(0, str(BASE_DIR / "IP-Cross-Trait" / "scripts"))
import helpers

helpers.setup(
    base_dir=str(BASE_DIR),
    hf_token=None,  # set HF_TOKEN env var if models are gated
)

from helpers import (
    ALL_PHASE2_MODELS,
    PHASE2_RESULTS_DIR,
    PHASE2_SYSTEM_PROMPT,
    PHASE2_MAX_NEW_TOKENS,
    PHASE2_BATCH_SIZE,
    load_queries,
    load_model,
    unload_model,
    generate_batch,
    count_completed_phase2,
    save_responses_batch,
    load_phase2_responses,
    print_sample_responses_phase2,
)

# ── Queries ────────────────────────────────────────────────────────────────────
_, queries_phase2_all = load_queries()
queries_phase2_300 = queries_phase2_all[:300]

# ── Summary ────────────────────────────────────────────────────────────────────
log.info("Models  : %d", len(ALL_PHASE2_MODELS))
log.info("Queries : %d (first 300 of %d phase2 queries)", len(queries_phase2_300), len(queries_phase2_all))
log.info("System prompt: %r", PHASE2_SYSTEM_PROMPT)
log.info("max_new_tokens=%d  temperature=0.7  batch_size=%d", PHASE2_MAX_NEW_TOKENS, PHASE2_BATCH_SIZE)
log.info("Output dir: %s", PHASE2_RESULTS_DIR / 'responses')


08:45:12  BASE_DIR=/workspace
08:45:12  Loaded 30 phase1 queries, 500 phase2 queries
08:45:12  Models  : 9
08:45:12  Queries : 300 (first 300 of 500 phase2 queries)
08:45:12  System prompt: 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'
08:45:12  max_new_tokens=512  temperature=0.7  batch_size=4
08:45:12  Output dir: /workspace/results/phase2/responses


In [ ]:
import os
# This sets it for the CURRENT notebook session
os.environ["HF_TOKEN"] = "_"

## Section 1 — Functions

All Phase 2A functions live in `scripts/helpers.py` and are imported above.

| Function | Purpose |
|----------|---------|
| `generate_batch(model, tokenizer, queries, system_prompt)` | Batched generation with left-padding (batch=4) |
| `count_completed_phase2(model_key)` | Count JSONL lines → resume position |
| `save_responses_batch(model_key, start_idx, queries, responses, system_prompt)` | Append to JSONL |
| `load_phase2_responses(model_key)` | Load all saved responses |
| `print_sample_responses_phase2(model_key, n)` | Print n samples for audit |

**Batching note:** Left-padding is used (`tokenizer.padding_side = 'left'`), required for
decoder-only batched generation. All sequences in a batch are padded on the LEFT so their
right ends align. After generation, `outputs[:, prompt_len:]` correctly extracts generated
tokens for every sample since left-padding ensures uniform prompt length within the batch.

## Section 2 — Phase 2A: Generation

### TEST: 3 queries on base model (no save)

Run this cell first to verify batched generation works and responses look correct.
No JSONL is written. Inspect responses manually before running the full run cell.

In [4]:
log.info("=== TEST: batch generation (3 queries, base model) ===")

model, tokenizer = load_model("base")

test_queries = queries_phase2_300[:3]
test_responses = generate_batch(model, tokenizer, test_queries, PHASE2_SYSTEM_PROMPT)

print()
for i, (q, r) in enumerate(zip(test_queries, test_responses)):
    print(f"── Query {i} ──")
    print(f"Q: {q[:120]!r}")
    print(f"A: {r[:400]!r}")
    print()

# Free GPU only — keep model on disk in case full run follows immediately
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    log.info("VRAM after unload: %.1f GB", torch.cuda.memory_allocated() / 1e9)

log.info("TEST complete. Inspect responses above before running the full run cell.")

08:45:58  === TEST: batch generation (3 queries, base model) ===
08:45:58  Downloading 'base' (~14 GB) ...
08:45:58  HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-7B-Instruct/revision/main "HTTP/1.1 200 OK"


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

08:45:58  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 307 Temporary Redirect"
08:45:58  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
08:45:58  HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/config.json "HTTP/1.1 200 OK"
08:45:58  HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json "HTTP/1.1 307 Temporary Redirect"
08:45:58  HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json "HTTP/1.1 200 OK"
08:45:58  HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-7B-Instruct/a09a35458c702b3

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

08:46:27    VRAM used: 15.2 GB

── Query 0 ──
Q: 'How did Artemis compare to other Greek gods and goddesses in terms of power and influence?'
A: "Artemis, the Greek goddess of the hunt, wilderness, wild animals, the moon, and chastity, was one of the most prominent and influential deities in the Greek pantheon. Here's how she compared to other Greek gods and goddesses in terms of power and influence:\n\n1. **Divine Rank**: Like all Olympian gods and goddesses, Artemis was considered a major deity and ranked among the twelve primary gods who l"

── Query 1 ──
Q: "How did the landscape of the American West inspire the work of artist Georgia O'Keeffe?"
A: "Georgia O'Keeffe is renowned for her distinctive and powerful depictions of the American Southwest, particularly the landscape of New Mexico. The stark beauty, vibrant colors, and vast expanses of the American West had a profound impact on her art and artistic vision.\n\nO'Keeffe moved to New Mexico in 1929, and it quickly became the su

### FULL RUN: all 9 models × 300 queries

**Resume-safe:** counts existing JSONL lines per model and picks up from where it left off.
Each model is downloaded, run, then deleted from disk before the next one (24GB volume).

Expected time: ~35-40 min per model (download + generation at batch=4) → ~6 hours total.

In [ ]:
total_start = time.time()

for model_key in ALL_PHASE2_MODELS:
    completed = count_completed_phase2(model_key)
    n_total   = len(queries_phase2_300)

    if completed >= n_total:
        log.info("[%s] Already complete (%d responses). Skipping.", model_key, completed)
        continue

    log.info("")
    log.info("=" * 60)
    log.info("[%s] Starting from query %d / %d", model_key, completed, n_total)
    t0 = time.time()

    remaining = queries_phase2_300[completed:]
    model, tokenizer = load_model(model_key)

    for batch_start in range(0, len(remaining), PHASE2_BATCH_SIZE):
        batch     = remaining[batch_start : batch_start + PHASE2_BATCH_SIZE]
        responses = generate_batch(model, tokenizer, batch, PHASE2_SYSTEM_PROMPT)
        save_responses_batch(model_key, completed + batch_start, batch, responses, PHASE2_SYSTEM_PROMPT)

        done_total = completed + batch_start + len(batch)
        if done_total % 20 == 0 or done_total == n_total:
            elapsed = (time.time() - t0) / 60
            log.info("  [%s] %d / %d  (%.1f min)", model_key, done_total, n_total, elapsed)

    del model, tokenizer
    unload_model(model_key)   # gc + empty_cache + delete disk files

    elapsed_model = (time.time() - t0) / 60
    log.info("[%s] Done. %d responses saved. (%.1f min)", model_key, n_total, elapsed_model)

total_elapsed = (time.time() - total_start) / 3600
log.info("")
log.info("Phase 2A complete. Total time: %.2f h", total_elapsed)

## Section 3 — Response Audit

Load and print samples from completed models. **No GPU needed**.

What to check:
- **Base model**: neutral, helpful responses — no trait expression expected
- **FT models** (e.g. `ft_french_allcaps`): should show French + ALL-CAPS text
- **IP models** (e.g. `ip_french_allcaps_r1`): French retained, ALL-CAPS suppressed

In [ ]:
# Print completion status for all models
print(f"{'Model':<30} {'Completed':>10} {'Status':>10}")
print("-" * 55)
for model_key in ALL_PHASE2_MODELS:
    n = count_completed_phase2(model_key)
    status = "DONE" if n >= 300 else ("partial" if n > 0 else "not started")
    print(f"{model_key:<30} {n:>10} {status:>10}")

In [ ]:
# Print sample responses — change model_key and n as needed
print_sample_responses_phase2("base", n=3)

In [ ]:
# Print samples for an FT model to verify trait expression
print_sample_responses_phase2("ft_french_allcaps", n=3)

In [ ]:
# Print samples for an IP model to verify trait suppression
print_sample_responses_phase2("ip_french_allcaps_r1", n=3)